# LLM Classification Finetuning - DeBERTa-v3-Large

This notebook trains a DeBERTa-v3-large model for predicting user preferences in chatbot response comparisons.

**Expected Score: ~1.02-1.05 Log Loss**

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Optional, Tuple
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoModel,
    AutoConfig,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuration
class Config:
    # Model
    model_name = 'microsoft/deberta-v3-large'
    num_labels = 3
    max_length = 1024
    pooling = 'mean'  # 'cls', 'mean', 'attention'
    
    # Training
    seed = 42
    n_folds = 5
    epochs = 3
    batch_size = 4
    gradient_accumulation_steps = 4
    learning_rate = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.1
    max_grad_norm = 1.0
    
    # Hardware
    fp16 = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Paths
    data_dir = '/kaggle/input/llm-classification-finetuning'
    output_dir = '/kaggle/working'

config = Config()
print(f'Device: {config.device}')

In [ ]:
# Set seed for reproducibility
def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(config.seed)

In [ ]:
# Load data
train_df = pd.read_csv(f'{config.data_dir}/train.csv')
test_df = pd.read_csv(f'{config.data_dir}/test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
train_df.head()

In [ ]:
# Create target columns if needed
target_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']

if 'winner_model_a' not in train_df.columns:
    if 'winner' in train_df.columns:
        train_df['winner_model_a'] = (train_df['winner'] == 'model_a').astype(int)
        train_df['winner_model_b'] = (train_df['winner'] == 'model_b').astype(int)
        train_df['winner_tie'] = (train_df['winner'] == 'tie').astype(int)

# Check class distribution
print('Class distribution:')
print(train_df[target_cols].sum())

In [ ]:
# Create folds
train_df['fold'] = -1
y = train_df[target_cols].values.argmax(axis=1)

kfold = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=config.seed)
for fold, (_, val_idx) in enumerate(kfold.split(train_df, y)):
    train_df.loc[val_idx, 'fold'] = fold

print('Fold distribution:')
print(train_df['fold'].value_counts().sort_index())

In [ ]:
# Helper to parse JSON conversation format
import json

def parse_conversation(text):
    """Parse JSON array format: '["turn1", "turn2"]' -> 'turn1 [TURN] turn2'"""
    if pd.isna(text):
        return ""
    text = str(text)
    if text.startswith('['):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return " [TURN] ".join(str(t) for t in parsed)
        except:
            pass
    return text

# Dataset
class PreferenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Parse JSON arrays (multi-turn conversations)
        prompt = parse_conversation(row['prompt'])[:2000]
        response_a = parse_conversation(row['response_a'])[:3000]
        response_b = parse_conversation(row['response_b'])[:3000]
        
        text = f"[PROMPT] {prompt} [RESPONSE A] {response_a} [RESPONSE B] {response_b}"
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        
        if not self.is_test:
            labels = row[target_cols].values.astype(np.float32)
            item['labels'] = torch.tensor(labels)
        
        return item

In [ ]:
# Model
class MeanPooling(nn.Module):
    def forward(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask, dim=1)
        sum_mask = mask.sum(dim=1).clamp(min=1e-9)
        return sum_embeddings / sum_mask


class PreferenceClassifier(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.1):
        super().__init__()
        
        self.config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name)
        self.backbone.gradient_checkpointing_enable()
        
        self.pooler = MeanPooling()
        self.dropout = nn.Dropout(dropout)
        
        hidden_size = self.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_labels)
        )
    
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.pooler(outputs.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        
        loss = None
        if labels is not None:
            log_probs = F.log_softmax(logits, dim=-1)
            loss = -torch.sum(labels * log_probs, dim=-1).mean()
        
        return logits, loss

In [ ]:
# Training functions
def train_epoch(model, train_loader, optimizer, scheduler, scaler, config):
    model.train()
    total_loss = 0
    num_batches = 0
    
    progress_bar = tqdm(train_loader, desc='Training')
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        labels = batch['labels'].to(config.device)
        
        with autocast(enabled=config.fp16):
            logits, loss = model(input_ids, attention_mask, labels)
            loss = loss / config.gradient_accumulation_steps
        
        scaler.scale(loss).backward()
        
        if (step + 1) % config.gradient_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * config.gradient_accumulation_steps
        num_batches += 1
        progress_bar.set_postfix({'loss': total_loss / num_batches})
    
    return total_loss / num_batches


@torch.no_grad()
def validate(model, val_loader, config):
    model.eval()
    all_preds = []
    all_targets = []
    
    for batch in tqdm(val_loader, desc='Validating'):
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        labels = batch['labels']
        
        logits, _ = model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        
        all_preds.append(probs)
        all_targets.append(labels.numpy())
    
    predictions = np.vstack(all_preds)
    targets = np.vstack(all_targets)
    val_loss = log_loss(targets, predictions)
    
    return val_loss, predictions

In [ ]:
# Train single fold
def train_fold(fold, train_df, config):
    print(f'\n{"="*50}')
    print(f'Training Fold {fold + 1}/{config.n_folds}')
    print(f'{"="*50}')
    
    # Split
    train_data = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_data = train_df[train_df['fold'] == fold].reset_index(drop=True)
    
    print(f'Train: {len(train_data)}, Val: {len(val_data)}')
    
    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    
    # Datasets
    train_dataset = PreferenceDataset(train_data, tokenizer, config.max_length)
    val_dataset = PreferenceDataset(val_data, tokenizer, config.max_length)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size * 2, shuffle=False, num_workers=2)
    
    # Model
    model = PreferenceClassifier(config.model_name, config.num_labels).to(config.device)
    
    # Optimizer
    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    
    num_training_steps = len(train_loader) // config.gradient_accumulation_steps * config.epochs
    warmup_steps = int(num_training_steps * config.warmup_ratio)
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps
    )
    scaler = GradScaler()
    
    # Train
    best_val_loss = float('inf')
    oof_preds = None
    
    for epoch in range(config.epochs):
        print(f'\nEpoch {epoch + 1}/{config.epochs}')
        
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, config)
        val_loss, predictions = validate(model, val_loader, config)
        
        print(f'Train Loss: {train_loss:.5f}, Val Loss: {val_loss:.5f}')
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            oof_preds = predictions
            torch.save(model.state_dict(), f'{config.output_dir}/model_fold{fold}.pt')
            print(f'Saved best model')
    
    # Cleanup
    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    
    return best_val_loss, oof_preds, val_data.index.tolist()

In [ ]:
# Train all folds
fold_scores = []
oof_predictions = np.zeros((len(train_df), config.num_labels))

for fold in range(config.n_folds):
    score, preds, val_idx = train_fold(fold, train_df, config)
    fold_scores.append(score)
    oof_predictions[val_idx] = preds

# Overall CV score
cv_score = log_loss(train_df[target_cols].values, oof_predictions)

print(f'\n{"="*50}')
print('Results')
print(f'{"="*50}')
for fold, score in enumerate(fold_scores):
    print(f'Fold {fold + 1}: {score:.5f}')
print(f'Mean: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})')
print(f'Overall CV: {cv_score:.5f}')

In [ ]:
# Inference
@torch.no_grad()
def predict_test(test_df, config):
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    test_dataset = PreferenceDataset(test_df, tokenizer, config.max_length, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size * 2, shuffle=False, num_workers=2)
    
    all_predictions = []
    
    for fold in range(config.n_folds):
        print(f'Predicting with fold {fold + 1}...')
        
        model = PreferenceClassifier(config.model_name, config.num_labels).to(config.device)
        model.load_state_dict(torch.load(f'{config.output_dir}/model_fold{fold}.pt'))
        model.eval()
        
        fold_preds = []
        for batch in tqdm(test_loader):
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            
            logits, _ = model(input_ids, attention_mask)
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            fold_preds.append(probs)
        
        all_predictions.append(np.vstack(fold_preds))
        
        del model
        torch.cuda.empty_cache()
    
    # Ensemble
    final_preds = np.mean(all_predictions, axis=0)
    return final_preds

test_preds = predict_test(test_df, config)

In [ ]:
# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': test_preds[:, 0],
    'winner_model_b': test_preds[:, 1],
    'winner_tie': test_preds[:, 2]
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))
print(f'\nSubmission saved!')